# 🩺 MONAI Label Server with Cloudflare Tunnel

This notebook sets up and runs a **MONAI Label** server in Google Colab, integrated with Google Drive for persistent storage. It also uses **Cloudflare Tunnel** to provide a secure, public URL so you can connect your local medical imaging software (like 3D Slicer) directly to this cloud-hosted AI server.

### 🚀 Key Features:
*   **MONAI Label Server:** AI-assisted labeling for medical imaging.
*   **Cloudflare Tunnel:** Exposes the server without complex port forwarding.
*   **Google Drive Integration:** Saves your data and trained models.
*   **Automated Setup:** Handles dependency conflicts (NumPy/pydicom) automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 📂 Project Directory Setup
This section defines the base directory in your Google Drive and creates the necessary subfolders for data, models, and logs. This ensures your project structure is organized and persistent.

In [ ]:
import os

BASE_DIR = "/content/drive/Shareddrives/Sandler and Kaminka/monai_server"

folders = [
    "data",
    "models",
    "transforms",
    "logs",
    "apps"
]

for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

print("Folders created under:", BASE_DIR)

Folders created under: /content/drive/Shareddrives/Sandler and Kaminka/monai_server


### 🛠 Environment Installation
This cell handles the installation of specific versions of NumPy and pydicom to avoid compatibility issues. It also installs the MONAI Label core package and downloads the Cloudflare tunnel binary.

In [ ]:
print("🔧 Updating environment for compatibility...")

# ---- 1. הסרת התקנות קודמות כדי למנוע שאריות בזיכרון ----
!pip uninstall -y numpy pydicom pydicom-seg monailabel pynetdicom >/dev/null 2>&1

# ---- 2. התקנה משולבת ----
# אנחנו מגדירים את הגרסאות הספציפיות שחיוניות למחקר שלך (PET/CT ו-Deep Learning)
# השימוש ב-monailabel ללא הגדרת גרסה ימשוך את האחרונה, אבל נאלץ את pydicom להישאר ב-2.4.4
!pip install -q numpy==1.26.4 \
                pydicom==2.4.4 \
                pydicom-seg==0.4.1 \
                antspyx==0.4.2 \
                monailabel

# ---- 3. Cloudflare tunnel ----
!wget -q -N https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import os
os.environ["CANCER_AI_ENV_READY"] = "1"

print("\n✅ Installation complete")
print("⚠️ IMPORTANT: Restart Runtime now (Runtime → Restart runtime)")
print("After restart, verify the version in the next cell.")

🔧 Updating environment for compatibility...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 req

### 🔄 Restart Runtime
Run the cell below to restart the Python kernel. This is necessary for the new package versions (like NumPy and pydicom) to take effect.

In [ ]:
import os
# This will restart the runtime automatically to apply the new library versions
os._exit(0)

### 🔍 Environment Verification
After restarting the runtime, run this cell to verify that the core libraries like NumPy and ANTs are correctly loaded. It ensures that the environment is stable before starting the server.

In [ ]:
# =========================================
# Environment Check
# Run this cell *after* restart the runtime
# =========================================

import numpy as np
import ants
import monailabel

print("NumPy:", np.__version__)
print("✅ Environment is working correctly")

NumPy: 1.26.4
✅ Environment is working correctly


### 🧬 MONAI Label App Selection
This section clones the official MONAI Label repository to access sample applications. We will specifically use the radiology app for this setup.

In [ ]:
import os
# Check if the repository already exists before cloning
if not os.path.exists('/content/MONAILabel'):
    !git clone https://github.com/Project-MONAI/MONAILabel.git
else:
    print("✅ MONAILabel repository already exists. Skipping clone.")

✅ MONAILabel repository already exists. Skipping clone.


### 🚀 Server Startup
This cell initializes the MONAI Label server in the background using `nohup`. It points the server to your Google Drive folders so that any progress is automatically saved.

In [ ]:
import os
import subprocess

# 1. Base Configuration (The real path with spaces)
BASE_DIR = "/content/drive/Shareddrives/Sandler and Kaminka/monai_server"

# 2. Create a Symlink without spaces
SYMLINK_DIR = "/content/monai_server_link"

if not os.path.exists(SYMLINK_DIR):
    os.symlink(BASE_DIR, SYMLINK_DIR)
    print(f"🔗 Created symlink: {SYMLINK_DIR} -> {BASE_DIR}")

# 3. Use the symlink for MONAI Label paths
# UPDATED: Pointing to 'dataset' as per your request
STUDIES_DIR = os.path.join(SYMLINK_DIR, "dataset")
MODEL_DIR = os.path.join(SYMLINK_DIR, "models")
APP_DIR = "/content/MONAILabel/sample-apps/radiology"

# Ensure directories exist
os.makedirs(os.path.join(BASE_DIR, "dataset"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

print("⌒ Starting MONAI Label server with dataset folder...")

# Construct command
cmd = f"nohup monailabel start_server --app {APP_DIR} --studies {STUDIES_DIR} --conf models segmentation --conf model_dir {MODEL_DIR} --port 8000 > server.log 2>&1 &"

print(f"Executing: {cmd}")
get_ipython().system(cmd)

print("\n✅ Server command sent. Please check server.log in 5-10 seconds.")

⌒ Starting MONAI Label server with dataset folder...
Executing: nohup monailabel start_server --app /content/MONAILabel/sample-apps/radiology --studies /content/monai_server_link/dataset --conf models segmentation --conf model_dir /content/monai_server_link/models --port 8000 > server.log 2>&1 &

✅ Server command sent. Please check server.log in 5-10 seconds.


### 📋 Debugging Logs
If the server fails to start, use this cell to read the logs. This helps identify if there are hidden dependency issues or path errors.

In [ ]:
import socket
import time

def check_server(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

print('Checking if MONAI Label server is active on port 8000...')
if check_server(8000):
    print('✅ Server is UP and listening!')
    print('You can now run the "Secure Tunneling" cell below to get your public URL.')
else:
    print('⏳ Server is still starting or failed. Wait 10 seconds and run this again.')
    print('If it stays down, check the full server.log for any "Traceback" errors.')

Checking if MONAI Label server is active on port 8000...
✅ Server is UP and listening!
You can now run the "Secure Tunneling" cell below to get your public URL.


### 🌐 Secure Tunneling
This script runs the Cloudflare tunnel to expose the local server to the internet. It will monitor the logs and display the unique public URL you need to enter into 3D Slicer.

In [ ]:
import subprocess
import time
import re

print("🌍 Starting Cloudflare tunnel...")

# Start cloudflared
proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Look for the URL in the output stream
url_found = False
for line in proc.stdout:
    # print(line.strip()) # Uncomment to see full debug logs
    if "trycloudflare.com" in line:
        url = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if url:
            print("\n" + "="*30)
            print("🔗 YOUR MONAI LABEL URL:")
            print(url.group(0))
            print("="*30)
            url_found = True
            break

if not url_found:
    print("❌ Could not find the tunnel URL. Please check if the server in the previous cell is actually running.")

🌍 Starting Cloudflare tunnel...

🔗 YOUR MONAI LABEL URL:
https://tried-emperor-comply-evaluation.trycloudflare.com


### 🛑 Shutdown
Use this cell when you are finished to safely stop both the MONAI Label server and the Cloudflare tunnel. This clears the background processes and releases the system resources.

In [ ]:
import os
import subprocess

print("🛑 Shutting down MONAI Label server and Cloudflare tunnel...")

# Kill monailabel and cloudflared processes
!pkill -f monailabel
!pkill -f cloudflared

print("✅ Processes stopped.")

🛑 Shutting down MONAI Label server and Cloudflare tunnel...
✅ Processes stopped.
